# Task 2, Step 3: Stage 1, Caption Alignment

Train only the connector for two passes over the 18,000 caption examples (2,250 updates, peak LR $10^{-3}$). Qwen's parameters stay frozen, but autograd stays on through its forward pass so that gradients reach the connector.

Before training, connector output is scaled to match text embedding magnitude, using training data only.

Loss is example-weighted causal cross-entropy, with visual tokens, prompts, and padding masked:
$$\mathcal{L}=-\frac{1}{|B|}\sum_{i\in B}\frac{1}{|R_i|}\sum_{t\in R_i}\log p(y_{i,t}\mid I_i,q_i,y_{i,<t})$$

In [ ]:
import os, sys, json
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "task2_vlm_grounding")):
    root = os.path.dirname(root)
sys.path.insert(0, root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from task2_vlm_grounding.src.notebook_utils import *
setup_notebook()

In [ ]:
from task2_vlm_grounding.src.notebook_utils import load_run
run = load_run()
history, dev, summary = run["history"], run["dev"], run["summary"]
scope = history[history["stage"] == 1]
if scope.empty:
    raise RuntimeError("Stage 1 has not been trained yet.")
print("Updates logged:", len(history), "of", summary["total_updates"], "| full recipe:", summary["full_recipe"])
print("Trainable parameters by stage:", summary["stage_params"], "| peak GPU MB:", round(summary["peak_gpu_mb"], 1))
print("Skipped updates:", len(summary["skipped_updates"]))
fig, axes = plt.subplots(1, 3, figsize=(17, 4.3))
axes[0].plot(scope["update"], scope["loss"], alpha=0.25, color="#3498db")
axes[0].plot(scope["update"], scope["loss"].rolling(25, min_periods=1).mean(), color="#2c3e50", label="25-update mean")
axes[0].set(title="Training loss (example-weighted)", xlabel="Main-run update", ylabel="Cross-entropy"); axes[0].legend()
axes[1].plot(scope["update"], scope["connector_lr"], label="Connector")
if scope["lora_lr"].notna().any():
    axes[1].plot(scope["update"], scope["lora_lr"], label="LoRA")
axes[1].set(title="Learning rates: 5% warmup, cosine to 10% of peak", xlabel="Main-run update", ylabel="LR", yscale="log"); axes[1].legend()
axes[2].plot(scope["update"], scope["grad_norm"], color="#e67e22", alpha=0.5)
axes[2].axhline(1.0, color="crimson", linestyle="--", label="Clip at 1.0")
axes[2].set(title="Gradient norm before clipping", xlabel="Main-run update", ylabel="Norm", yscale="log"); axes[2].legend()
save_figure("task2_stage1_training")

## 3.1 Calibration and development caption loss

In [ ]:
with open(f"{RESULTS_DIR}/main/calibration.json") as f:
    display(pd.DataFrame([json.load(f)]))
stage_dev = dev[dev["update"] <= 2250]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(stage_dev["update"], stage_dev["caption_loss"], marker="o", color="#2980b9")
ax.set(title="Development caption reference loss", xlabel="Main-run update", ylabel="Cross-entropy")
save_figure("task2_stage1_dev_caption")
display(stage_dev[["update", "caption_loss"]].round(4))